# Step 2 — 01. Preprocessing and model smoke test

등록된 factory로 실제 PyTorch 모델을 만들고 고정된 정렬 crop에서
raw 512D, raw norm, L2-normalized embedding 및 Grad-CAM target
layer를 검증합니다. 입력 파일을 지정하지 않으면 LFW deep-funneled
manifest에서 identity가 겹치지 않는 소량 이미지를 결정적으로 골라
112×112 smoke 입력을 자동 생성합니다. 이 입력은 모델 동작 확인 전용이며
정량 실험이나 Grad-CAM population 입력으로 재사용하지 않습니다.

In [ ]:
# cell 1 : 환경 설정 및 profile 선택
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("research와 configs가 있는 프로젝트 루트를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

# ─── Profile 선택 ──────────────────────────────────────────────────
MODEL_PROFILE = "arcface_ms1mv3_r100"  # 00에서 등록한 profile과 동일하게 설정
# ──────────────────────────────────────────────────────────────────

MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 1          # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False
WRITE_OUTPUTS = False

# Profile 유효성 검증
all_profiles = (
    CONFIG["models"]["selected_profiles"]
    + CONFIG["models"].get("bridge_profiles", [])
)
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    raise RuntimeError(f"차단된 profile입니다: {MODEL_PROFILE}")

if MODEL_PROFILE not in available_profiles:
    raise ValueError(
        f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}\n"
        f"사용 가능한 profile: {sorted(available_profiles)}"
    )

PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]

if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

print(f"Profile: {MODEL_PROFILE} ({MODEL_FAMILY}, {PROFILE['training_dataset']})")

In [ ]:
# cell 2 : 모델 로딩 및 smoke test 설정
import json
import numpy as np

from research.embeddings import (
    create_pytorch_adapter_from_spec,
    resolve_smoke_input_batch,
    select_model_spec,
    select_model_spec_by_profile,
)

MODEL_REGISTRY_ROOT = PROJECT_ROOT / "runs/step2/model_registry"
MODEL_UID = None             # 같은 family에 2개 이상의 spec이 있을 때만 정확한 model_uid 지정
SMOKE_INPUT_PATH = None      # 선택: 기존 .npy/.npz. None이면 LFW에서 자동 생성
SMOKE_MANIFEST_PATH = PROJECT_ROOT / CONFIG["datasets"]["lfw"]["manifest_path"]
DEVICE = "cpu"               # GPU 확인 후 "cuda"로 변경 가능
MAX_SMOKE_IMAGES = 8

In [ ]:
# cell 3 : smoke test 실행
if EXECUTE_STAGE:
    if not MODEL_REGISTRY_ROOT.is_dir():
        raise RuntimeError(
            "등록된 ModelSpec이 없습니다. 먼저 "
            "00_checkpoint_registration.ipynb에서 공식 checkpoint를 "
            "등록하고 WRITE_OUTPUTS=True로 실행하세요."
        )
    # Profile 기반 선택: family + architecture + training_dataset 일치 확인
    try:
        model_spec_path, spec = select_model_spec_by_profile(
            MODEL_REGISTRY_ROOT,
            profile_id=MODEL_PROFILE,
            profile_config=PROFILE,
            verify_checkpoint=True,
        )
    except Exception:
        # fallback: 기존 family 기반 선택 (하위 호환)
        model_spec_path, spec = select_model_spec(
            MODEL_REGISTRY_ROOT,
            family=MODEL_FAMILY,
            model_uid=MODEL_UID,
            verify_checkpoint=True,
        )
    smoke_input = resolve_smoke_input_batch(
        PROJECT_ROOT,
        source_color_order=spec.preprocessing.source_color_order,
        explicit_path=SMOKE_INPUT_PATH,
        lfw_manifest_path=SMOKE_MANIFEST_PATH,
        max_images=MAX_SMOKE_IMAGES,
        seed=SEED,
    )
    adapter = create_pytorch_adapter_from_spec(spec, device=DEVICE)
    aligned_faces = smoke_input.aligned_faces
    output = adapter.embed(aligned_faces)
    _ = adapter.target_layer

    unit_norms = np.linalg.norm(output.normalized_embedding, axis=1)
    smoke_summary = {
        "model_uid": spec.model_uid,
        "profile_id": MODEL_PROFILE,
        "family": spec.family,
        "architecture": spec.architecture,
        "training_dataset": spec.training_dataset,
        "model_spec_path": str(model_spec_path),
        "smoke_input": smoke_input.metadata,
        "checkpoint_sha256": spec.checkpoint.sha256,
        "preprocess_hash": spec.preprocessing.preprocess_hash,
        "sample_count": int(len(aligned_faces)),
        "raw_shape": list(output.raw_embedding.shape),
        "raw_norm_min": float(output.raw_norm.min()),
        "raw_norm_max": float(output.raw_norm.max()),
        "maximum_unit_norm_error": float(np.max(np.abs(unit_norms - 1.0))),
        "target_layer": spec.target_layer,
        "status": "validated",
    }
    if WRITE_OUTPUTS:
        destination = (
            PROJECT_ROOT
            / "runs/step2/model_validation"
            / spec.model_uid
            / "smoke_summary.json"
        )
        if destination.exists():
            raise FileExistsError(f"기존 결과를 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(
            json.dumps(smoke_summary, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
else:
    smoke_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
        "profile_id": MODEL_PROFILE,
    }
smoke_summary

이 smoke test가 통과해도 세 loss의 인과 비교가 성립하는 것은
아닙니다. 이후 정량 실험은 모델별 새 embedding/PCA/PQ lineage에서
수행해야 하며 Step 1 ONNX artifact에 합치지 않습니다.